# OmniVoice Robust Long-Form TTS

Colab workflow for the `fix/robust-longform-tts` branch. It includes the silence-edge fix from PR #259 plus semantic chunking, ASR verification, retry, and recursive split for failed chunks.

Select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a T4 GPU first.'

In [ ]:
!pip install -q 'git+https://github.com/binhminhanh1235/OmniVoice.git@fix/robust-longform-tts' soundfile


In [ ]:
from google.colab import files
uploaded = files.upload()
REF_AUDIO = next(iter(uploaded))
print('Reference:', REF_AUDIO)

In [ ]:
REF_TEXT = '''PASTE THE EXACT REFERENCE TRANSCRIPT HERE.'''.strip()

TEXT = '''
Let’s be clear from the beginning.

This is not about refusing kindness to someone who is sick, grieving, poor, overwhelmed, or genuinely trying to rebuild their life.

This is not about becoming suspicious of everyone who needs help.

And this is not about calling people “toxic” because they disappointed you.

This is about repeated patterns.

Patterns that reject truth.
Patterns that avoid responsibility.
Patterns that turn compassion into permission.

Love is not unlimited access.

Forgiveness is not instant trust.

And helping a person is not the same as helping the pattern that keeps hurting them, others, and sometimes you.

So as we walk through these five patterns, do not use them to judge someone quickly. Use them first to examine the kind of help you are giving.
'''.strip()

assert 'PASTE THE EXACT' not in REF_TEXT, 'Set REF_TEXT before generating.'

In [ ]:
import torch
from omnivoice import (
    OmniVoice, OmniVoiceGenerationConfig,
    RobustLongFormConfig, RobustLongFormGenerator,
)

model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice',
    device_map='cuda:0',
    dtype=torch.float16,
)

voice_prompt = model.create_voice_clone_prompt(
    ref_audio=REF_AUDIO,
    ref_text=REF_TEXT,
    preprocess_prompt=True,
)

robust = RobustLongFormGenerator(
    model,
    RobustLongFormConfig(
        max_chunk_words=28,
        max_retries=3,
        max_split_depth=2,
        verify_with_asr=True,
        asr_model_name='openai/whisper-small.en',
        asr_device='cpu',
        strict=False,
    ),
)

gen_config = OmniVoiceGenerationConfig(
    num_step=32,
    guidance_scale=2.0,
    position_temperature=1.0,
    class_temperature=0.0,
)

In [ ]:
result = robust.generate(
    TEXT,
    language='en',
    voice_clone_prompt=voice_prompt,
    generation_config=gen_config,
)

print('All verified:', result.all_verified)
for i, report in enumerate(result.reports, 1):
    print(f'{i:02d} PASS={report.accepted} WER={report.wer:.3f} attempts={report.attempts}')
    print(' expected:', report.text)
    print(' ASR     :', report.transcript)

In [ ]:
import soundfile as sf
from IPython.display import Audio, display
from google.colab import files

OUTPUT = '/content/omnivoice_robust_output.wav'
sf.write(OUTPUT, result.audio, result.sampling_rate)
display(Audio(OUTPUT))
files.download(OUTPUT)